[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
import torch
import math

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    _, seq_q, d_k = Q.shape
    _, seq_k, d_k = K.shape
    mask = torch.tril(torch.ones(seq_q, seq_k)).bool()
    attn_score = torch.einsum('...qd,...kd->...qk',Q,K)/math.sqrt(d_k)
    attn_score = attn_score.masked_fill(~mask,-float('inf'))
    return torch.einsum('...qk,...kd->...qd',torch.softmax(attn_score,dim=-1),V)

In [11]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [12]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (37.6ms)
  ✅ [2/4] Future tokens don't affect past (4.6ms)
  ✅ [3/4] First position only sees itself (0.5ms)
  ✅ [4/4] Gradient flow (5.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (47.8ms total)
  Progress saved. Run status() to see your dashboard.

